### Importing necessary libraries

In [ ]:
import pandas as pd
import numpy
import sqlite3 as sql
import datetime

!pip install peewee

from peewee import *
import os

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
try:
    os.remove('loan_applications.db')
except OSError:
    pass

The database file is deleted to ensure a clean slate each time to avoid any issues regarding tables/views already existing and ensuring the data is properly cleaned and added each time.

### Importing data

In [ ]:
data_card= pd.read_csv('card.csv', sep=';')

In [ ]:
data_client= pd.read_csv('client.csv', sep=';')

In [ ]:
data_loans = pd.read_csv('loan.csv', sep=';')

In [ ]:
data_orders = pd.read_csv('order.csv', sep=';')

In [ ]:
data_district = pd.read_csv('district.csv', sep=';')

In [ ]:
data_disp = pd.read_csv('disp.csv', sep=';')

In [ ]:
data_trans =pd.read_csv('trans.csv',sep=';')

/usr/local/lib/python3.8/dist-packages/IPython/core/interactiveshell.py:3326: DtypeWarning: Columns (8) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [ ]:
data_account =pd.read_csv('account.csv',sep=';')

### Defining functions for cleaning data

In [ ]:
# Create a function to convert the date value to the 20th century

def add_prefix(int_value):
    return "19" + str(int_value)

In [ ]:
# Create a function to deducts 50 from women's birth months in order to clean data
def convert_month(date_int):
    try:
        # Convert the integer to a string
        date_str = str(date_int)

        month_digits = date_str[4:6]
        month_int = int(month_digits)

        # Check if the month is greater than or equal to 51
        if month_int >= 51:
            # Subtract 50 from the month
            date_str = date_str[:4] + str(int(date_str[4:6]) - 50) + date_str[6:]

        return date_str
    except ValueError:
        # Return None if the string cannot be converted
        return None

In [ ]:
# Create a function that converts the date integer to date string
def convert_to_date(date_int):
    try:
        # Convert the integer to a string
        date_str = str(date_int)

        return date_str
    except ValueError:
        # Return None if the string cannot be converted
        return None

In [ ]:
# Remove time from datetime
def to_only_yyyymmdd(date_str):
    try:

        date_dt = date_str[:6]

        return date_dt

    except ValueError:
        # Return None if the string cannot be converted
        return None

### Cleaning data

#### Clients

In [ ]:
data_client['birth_number'] = data_client['birth_number'].apply(add_prefix)

In [ ]:
data_client['birth_number'] = data_client['birth_number'].apply(convert_month)

#### Transactions

In [ ]:
data_trans['date'] = data_trans['date'].apply(add_prefix)

In [ ]:
data_trans['date'] = data_trans['date'].apply(convert_to_date)

#### Accounts

In [ ]:
data_account['date'] = data_account['date'].apply(add_prefix)

In [ ]:
data_account['date'] = data_account['date'].apply(convert_to_date)

#### Loans

In [ ]:
data_loans['date'] = data_loans['date'].apply(add_prefix)

In [ ]:
data_loans['date'] = data_loans['date'].apply(convert_to_date)

#### Credit cards

In [ ]:
data_card['issued'] = data_card['issued'].apply(add_prefix)

In [ ]:
data_card['issued'] = data_card['issued'].apply(to_only_yyyymmdd)

In [ ]:
data_card['issued'] = data_card['issued'].apply(convert_to_date)

### View tables

In [ ]:
data_card

,card_id,disp_id,type,issued
0,1005,9285,classic,199311
1,104,588,classic,199401
2,747,4915,classic,199402
3,70,439,classic,199402
4,577,3687,classic,199402
...,...,...,...,...
887,125,694,gold,199812
888,674,4360,classic,199812
889,322,2063,classic,199812
890,685,4467,classic,199812


In [ ]:
data_client

,client_id,birth_number,district_id
0,1,19701213,18
1,2,19450204,1
2,3,19401009,1
3,4,19561201,5
4,5,1960703,5
...,...,...,...
5364,13955,19451030,1
5365,13956,19430406,1
5366,13968,19680413,61
5367,13971,19621019,67


In [ ]:
data_loans

,loan_id,account_id,date,amount,duration,payments,status
0,5314,1787,19930705,96396,12,8033.0,B
1,5316,1801,19930711,165960,36,4610.0,A
2,6863,9188,19930728,127080,60,2118.0,A
3,5325,1843,19930803,105804,36,2939.0,A
4,7240,11013,19930906,274740,60,4579.0,A
...,...,...,...,...,...,...,...
677,4989,105,19981205,352704,48,7348.0,C
678,5221,1284,19981205,52512,12,4376.0,C
679,6402,6922,19981206,139488,24,5812.0,C
680,5346,1928,19981206,55632,24,2318.0,C


In [ ]:
data_orders

,order_id,account_id,bank_to,account_to,amount,k_symbol
0,29401,1,YZ,87144583,2452.0,SIPO
1,29402,2,ST,89597016,3372.7,UVER
2,29403,2,QR,13943797,7266.0,SIPO
3,29404,3,WX,83084338,1135.0,SIPO
4,29405,3,CD,24485939,327.0,
...,...,...,...,...,...,...
6466,46334,11362,YZ,70641225,4780.0,SIPO
6467,46335,11362,MN,78507822,56.0,
6468,46336,11362,ST,40799850,330.0,POJISTNE
6469,46337,11362,KL,20009470,129.0,


In [ ]:
data_district

,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11,A12,A13,A14,A15,A16
0,1,Hl.m. Praha,Prague,1204953,0,0,0,1,1,100.0,12541,0.29,0.43,167,85677,99107
1,2,Benesov,central Bohemia,88884,80,26,6,2,5,46.7,8507,1.67,1.85,132,2159,2674
2,3,Beroun,central Bohemia,75232,55,26,4,1,5,41.7,8980,1.95,2.21,111,2824,2813
3,4,Kladno,central Bohemia,149893,63,29,6,2,6,67.4,9753,4.64,5.05,109,5244,5892
4,5,Kolin,central Bohemia,95616,65,30,4,1,6,51.4,9307,3.85,4.43,118,2616,3040
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72,73,Opava,north Moravia,182027,17,49,12,2,7,56.4,8746,3.33,3.74,90,4355,4433
73,74,Ostrava - mesto,north Moravia,323870,0,0,0,1,1,100.0,10673,4.75,5.44,100,18782,18347
74,75,Prerov,north Moravia,138032,67,30,4,2,5,64.6,8819,5.38,5.66,99,4063,4505
75,76,Sumperk,north Moravia,127369,31,32,13,2,7,51.2,8369,4.73,5.88,107,3736,2807


In [ ]:
data_disp

,disp_id,client_id,account_id,type
0,1,1,1,OWNER
1,2,2,2,OWNER
2,3,3,2,DISPONENT
3,4,4,3,OWNER
4,5,5,3,DISPONENT
...,...,...,...,...
5364,13647,13955,11349,OWNER
5365,13648,13956,11349,DISPONENT
5366,13660,13968,11359,OWNER
5367,13663,13971,11362,OWNER


In [ ]:
data_trans

,trans_id,account_id,date,type,operation,amount,balance,k_symbol,bank,account
0,695247,2378,19930101,PRIJEM,VKLAD,700.0,700.0,NaN,NaN,NaN
1,171812,576,19930101,PRIJEM,VKLAD,900.0,900.0,NaN,NaN,NaN
2,207264,704,19930101,PRIJEM,VKLAD,1000.0,1000.0,NaN,NaN,NaN
3,1117247,3818,19930101,PRIJEM,VKLAD,600.0,600.0,NaN,NaN,NaN
4,579373,1972,19930102,PRIJEM,VKLAD,400.0,400.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
1056315,3626622,2906,19981231,PRIJEM,NaN,62.3,13729.4,UROK,NaN,NaN
1056316,3627616,2935,19981231,PRIJEM,NaN,81.3,19544.9,UROK,NaN,NaN
1056317,3625403,2869,19981231,PRIJEM,NaN,60.2,14638.2,UROK,NaN,NaN
1056318,3626683,2907,19981231,PRIJEM,NaN,107.5,23453.0,UROK,NaN,NaN


In [ ]:
data_account

,account_id,district_id,frequency,date
0,576,55,POPLATEK MESICNE,19930101
1,3818,74,POPLATEK MESICNE,19930101
2,704,55,POPLATEK MESICNE,19930101
3,2378,16,POPLATEK MESICNE,19930101
4,2632,24,POPLATEK MESICNE,19930102
...,...,...,...,...
4495,124,55,POPLATEK MESICNE,19971228
4496,3958,59,POPLATEK MESICNE,19971228
4497,777,30,POPLATEK MESICNE,19971228
4498,1573,63,POPLATEK MESICNE,19971229


### Connecting to the database

In [ ]:
db = SqliteDatabase('loan_applications.db')

In [ ]:
database = "loan_applications.db"
conn = sql.connect(database)
cursor = conn.cursor()

### Creating tables

In [ ]:
class Districts(Model):
    district_id = IntegerField(primary_key=True)
    district_name = TextField()
    region = TextField()
    pop = IntegerField()
    pop_less_499 = IntegerField()
    pop_less_1999 = IntegerField()
    pop_less_9999 = IntegerField()
    pop_greater_10000 = IntegerField()
    city_no = IntegerField()
    urb_ratio = DecimalField()
    avg_sal = DecimalField()
    unemployment_95 = DecimalField()
    unemployment_96 = DecimalField()
    entreprenuers_per_1000 = DecimalField()
    crimes_95 = DecimalField()
    crimes_96 = DecimalField()

    class Meta:
        database = db

class Accounts(Model):
    account_id = IntegerField(primary_key=True)
    district_id = ForeignKeyField(Districts, to_field = 'district_id')
    frequency = TextField()
    date = DateField()

    class Meta:
        database = db

class Clients(Model):
    client_id = IntegerField(primary_key = True)
    birth_number = DateField()
    district_id = ForeignKeyField(Districts, to_field = 'district_id')

    class Meta:
        database = db

class Dispositions(Model):
    disp_id = IntegerField(primary_key=True)
    client_id = ForeignKeyField(Clients, to_field = 'client_id')
    account_id = ForeignKeyField(Accounts, to_field = 'account_id')
    disp_type = TextField(constraints = [Check("disp_type IN ('OWNER', 'DISPONENT')")])

    class Meta:
        database = db

class Credit_cards(Model):
    card_id = IntegerField(primary_key = True)
    disp_id = ForeignKeyField(Dispositions, to_field = 'disp_id')
    card_type = TextField(constraints = [Check("card_type IN ('junior', 'classic', 'gold')")])
    issued = DateField()

    class Meta:
        database = db

class Loans(Model):
    loan_id = IntegerField(primary_key=True)
    account_id = ForeignKeyField(Accounts, to_field = 'account_id')
    date = DateField()
    amount = IntegerField()
    duration = IntegerField()
    payments = DecimalField()
    status = TextField(constraints = [Check("status IN ('A', 'B', 'C', 'D')")])

    class Meta:
        database = db

class Payment_orders(Model):
    order_id = IntegerField(primary_key=True)
    account_id = ForeignKeyField(Accounts, to_field = 'account_id')
    bank_to = TextField()
    account_to = IntegerField()
    amount = DecimalField()
    k_symbol = TextField(null = True)

    class Meta:
        database = db

class Transactions(Model):
    trans_id = IntegerField(primary_key=True)
    account_id = ForeignKeyField(Accounts, to_field = 'account_id')
    date = DateField()
    trans_type = TextField()
    operation = TextField(null = True)
    amount = DecimalField()
    balance = DecimalField()

    class Meta:
        database = db




In [ ]:
db.create_tables([Credit_cards,Clients,Loans,Payment_orders,Loans,Districts,Dispositions,Transactions,Accounts])

### Populating tables with data

In [ ]:
for row in data_card.itertuples():
    cursor.execute('''
                INSERT INTO Credit_cards (card_id, disp_id, card_type, issued)
                VALUES (?,?,?,?)
                ''',
                (row.card_id,
                row.disp_id,
                row.type,
                row.issued)
                )

conn.commit()

In [ ]:
for row in data_client.itertuples():
    cursor.execute('''
                INSERT INTO Clients (client_id, birth_number, district_id)
                VALUES (?,?,?)
                ''',
                (row.client_id,
                row.birth_number,
                row.district_id)
                )

conn.commit()

In [ ]:
for row in data_loans.itertuples():
    cursor.execute('''
                INSERT INTO Loans (loan_id, account_id, date, amount, duration, payments, status)
                VALUES (?,?,?,?,?,?,?)
                ''',
                (row.loan_id,
                row.account_id,
                row.date,
                row.amount,
                row.duration,
                row.payments,
                row.status)
                )

conn.commit()

In [ ]:
for row in data_orders.itertuples():
    cursor.execute('''
                INSERT INTO Payment_orders (order_id, account_id, bank_to, account_to, amount, k_symbol)
                VALUES (?,?,?,?,?,?)
                ''',
                (row.order_id,
                row.account_id,
                row.bank_to,
                row.account_to,
                row.amount,
                row.k_symbol)
                )

conn.commit()

In [ ]:
for row in data_district.itertuples():
    cursor.execute('''
                INSERT INTO Districts (district_id, district_name, region, pop, pop_less_499, pop_less_1999, pop_less_9999,pop_greater_10000,
                city_no, urb_ratio, avg_sal, unemployment_95, unemployment_96, entreprenuers_per_1000, crimes_95, crimes_96)
                VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
                ''',
                (row.A1,
                row.A2,
                row.A3,
                row.A4,
                row.A5,
                row.A6,
                row.A7,
                row.A8,
                row.A9,
                row.A10,
                row.A11,
                row.A12,
                row.A13,
                row.A14,
                row.A15,
                row.A16)
                )

conn.commit()

In [ ]:
for row in data_disp.itertuples():
    cursor.execute('''
                INSERT INTO Dispositions (disp_id, client_id, account_id, disp_type)
                VALUES (?,?,?,?)
                ''',
                (row.disp_id,
                row.client_id,
                row.account_id,
                row.type)
                )

conn.commit()

In [ ]:
for row in data_trans.itertuples():
    cursor.execute('''
                INSERT INTO Transactions (trans_id, account_id, date, trans_type,
                operation, amount, balance)
                VALUES (?,?,?,?,?,?,?)
                ''',
                (row.trans_id,
                row.account_id,
                row.date,
                row.type,
                row.operation,
                row.amount,
                row.balance)
                )

conn.commit()

In [ ]:
for row in data_account.itertuples():
    cursor.execute('''
                INSERT INTO Accounts (account_id, district_id, frequency, date)
                VALUES (?,?,?,?)
                ''',
                (row.account_id,
                row.district_id,
                row.frequency,
                row.date)
                )

conn.commit()

In [ ]:
tables = db.get_tables()
for table in tables:
    print('Table:',table,'Primary Key:',db.get_primary_keys(table))

Table: accounts Primary Key: ['account_id']
Table: clients Primary Key: ['client_id']
Table: credit_cards Primary Key: ['card_id']
Table: dispositions Primary Key: ['disp_id']
Table: districts Primary Key: ['district_id']
Table: loans Primary Key: ['loan_id']
Table: payment_orders Primary Key: ['order_id']
Table: transactions Primary Key: ['trans_id']


Validating Primary keys have been successfully constructed

In [ ]:
for table in tables:
    print('Table:',table)
    print(db.get_foreign_keys(table))
    print()

Table: accounts
[ForeignKeyMetadata(column='district_id', dest_table='districts', dest_column='district_id', table='accounts')]

Table: clients
[ForeignKeyMetadata(column='district_id', dest_table='districts', dest_column='district_id', table='clients')]

Table: credit_cards
[ForeignKeyMetadata(column='disp_id', dest_table='dispositions', dest_column='disp_id', table='credit_cards')]

Table: dispositions
[ForeignKeyMetadata(column='account_id', dest_table='accounts', dest_column='account_id', table='dispositions'), ForeignKeyMetadata(column='client_id', dest_table='clients', dest_column='client_id', table='dispositions')]

Table: districts
[]

Table: loans
[ForeignKeyMetadata(column='account_id', dest_table='accounts', dest_column='account_id', table='loans')]

Table: payment_orders
[ForeignKeyMetadata(column='account_id', dest_table='accounts', dest_column='account_id', table='payment_orders')]

Table: transactions
[ForeignKeyMetadata(column='account_id', dest_table='accounts', dest_c

Validating Foreign keys have been correctly cast in each table

### Creating a log trigger

**Purpose:** This trigger keeps a log of any updates in the 'Loans' table by inserting the value that was changed, the value that was changed to and the timestamp of when the update was done. This trigger is helpful for managing the database.<br>
**Input parameters and conditions:** There are no input parameters or conditions. <br>
**Expected outputs:** There are no expected outputs.

In [ ]:
script = '''

CREATE TABLE log (
    loan_id INTEGER NOT NULL UNIQUE,
    column_name VARCHAR(255) NOT NULL,
    old_value VARCHAR(255) NOT NULL,
    new_value VARCHAR(255) NOT NULL
);

'''

In [ ]:
conn.execute(script)

In [ ]:
script = '''

CREATE TRIGGER update_log
AFTER UPDATE ON Loans
FOR EACH ROW
BEGIN

    /* Insert a row into the log table with the old and new values of the updated row, and the current timestamp */

    INSERT INTO log(loan_id, column_name, old_value, new_value, timestamp)
    VALUES (OLD.loan_id, 'amount', OLD.amount, NEW.amount, CURRENT_TIMESTAMP),
           (OLD.loan_id, 'duration', OLD.loan_duration, NEW.duration, CURRENT_TIMESTAMP),
           (OLD.loan_id, 'payments', OLD.payments, NEW.payments, CURRENT_TIMESTAMP);
END

'''

In [ ]:
conn.execute(script)

# QUERIES

### QUERY 1: List the districts in descending order according to how many clients they have

**Purpose:** With this query we can list all the districts in a descending order from the 'Distrcits' table according to the number of clients they have. This is useful for the bank to understand where their clients are based which could for example help them decide to which branches send more staff to provide any services. <br>
**Input parameters and conditions:** There is no input parameters or conditions.<br>
**Expected outputs:** In this query, the expected outputs are the column 'district_id', the count of 'district_id' from the 'Clients' table and 'district_name', 'region' from the 'Districts' table. The expected output is ordered in a descending order by the count of 'district_id' <br>

In [ ]:
script = '''

WITH district_count AS
    (SELECT district_id, COUNT(district_id) AS count_distr
        FROM Clients
        GROUP BY district_id),
        district_list AS
        (SELECT DC.district_id, D.district_name, D.region, DC.count_distr
        FROM Districts AS D
        LEFT JOIN district_count AS DC ON DC.district_id = D.district_id)


SELECT * FROM district_list
ORDER BY count_distr DESC;

'''

In [ ]:
result = conn.execute(script)
for row in result:
    print(row)

(1, 'Hl.m. Praha', 'Prague', 663)
(74, 'Ostrava - mesto', 'north Moravia', 180)
(70, 'Karvina', 'north Moravia', 169)
(54, 'Brno - mesto', 'south Moravia', 155)
(64, 'Zlin', 'south Moravia', 109)
(72, 'Olomouc', 'north Moravia', 104)
(68, 'Frydek - Mistek', 'north Moravia', 86)
(46, 'Nachod', 'east Bohemia', 76)
(52, 'Usti nad Orlici', 'east Bohemia', 73)
(5, 'Kolin', 'central Bohemia', 71)
(8, 'Mlada Boleslav', 'central Bohemia', 69)
(33, 'Decin', 'north Bohemia', 69)
(36, 'Liberec', 'north Bohemia', 67)
(19, 'Prachatice', 'south Bohemia', 66)
(66, 'Zdar nad Sazavou', 'south Moravia', 66)
(38, 'Louny', 'north Bohemia', 65)
(59, 'Kromeriz', 'south Moravia', 64)
(3, 'Beroun', 'central Bohemia', 63)
(15, 'Cesky Krumlov', 'south Bohemia', 63)
(47, 'Pardubice', 'east Bohemia', 63)
(55, 'Brno - venkov', 'south Moravia', 63)
(16, 'Jindrichuv Hradec', 'south Bohemia', 61)
(50, 'Svitavy', 'east Bohemia', 61)
(51, 'Trutnov', 'east Bohemia', 61)
(60, 'Prostejov', 'south Moravia', 61)
(69, 'Jesen

### QUERY 2: Count how many different credit cards there are

**Purpose:** This query lets the database user know how the number of credit card users distributes between different types of credit cards. This could be helpful, for example, for the bank to understand which credit card types are more popular which is useful in developing new credit card products or marketing strategies.  <br>
**Input parameters and conditions:** There is no input parameters or conditions.<br>
**Expected outputs:** The expected output includes the 'type' column the 'Credit_cards' table and the count for that column. The results are order in a descending order by the count for different types of credit cards.<br>

In [ ]:
script = '''

SELECT card_type, COUNT(card_type) AS count_type
FROM Credit_cards
GROUP BY card_type
ORDER BY count_type DESC

'''

In [ ]:
result = conn.execute(script)
for row in result:
    print(row)

('classic', 659)
('junior', 145)
('gold', 88)


### QUERY 3: List the average salary and average yearly ? (amount) of a loan for each district

**Purpose:** This query gives the bank a good comparison of each district's average salary and average yearly amount of loan. Bank can use this information on decisions whether to grant on a loan application and to ensure that granted loans are sustainable and successful as comparing the average salary in a district with the average yearly loan amount, gives a better sense of the borrower's financial situation any potential risks when lending to them.    <br>
**Input parameters and conditions:** There is no input parameters or conditions.<br>
**Expected outputs:** Expected output includes the 'district_id' column from the 'Accounts' table which previously has been LEFT JOINED with the 'Loans' table, 'avg_sal' column from 'Districts' table and the average yearly loan for a relevant district. <br>

In [ ]:
script = '''


WITH loans_list AS
    (SELECT * FROM Loans),

    loans_district AS
        (SELECT LL.account_id, LL.loan_id, LL.amount, LL.duration, A.district_id
        FROM loans_list as LL
        LEFT JOIN Accounts AS A ON LL.account_id = A.account_id),

       loans_calculation AS
           (SELECT account_id, loan_id, district_id, amount * 12 / duration AS loan_year
            FROM loans_district),

        district_average_loan AS
            (SELECT district_id, AVG(loan_year) as average_loan
            FROM loans_calculation
            GROUP BY district_id),

        district_info AS
            (SELECT DAL.district_id, D.avg_sal, DAL.average_loan
            FROM Districts as D
            LEFT JOIN district_average_loan AS DAL ON D.district_id = DAL.district_id)


SELECT * FROM district_info
ORDER BY avg_sal DESC'''

In [ ]:
result = conn.execute(script)
for row in result:
    print(row)

(1, 12541, 52067.42857142857)
(8, 11277, 50124.0)
(26, 10787, 32608.0)
(74, 10673, 58908.0)
(39, 10446, 40178.4)
(70, 10177, 54083.0)
(10, 10124, 53210.4)
(14, 10045, 64285.5)
(7, 9920, 53202.0)
(54, 9897, 49003.5)
(68, 9893, 44850.75)
(41, 9832, 51064.0)
(4, 9753, 42958.28571428572)
(34, 9675, 47393.333333333336)
(30, 9650, 61230.0)
(64, 9624, 54037.41176470588)
(11, 9622, 51855.27272727273)
(47, 9538, 44368.8)
(43, 9425, 39900.0)
(40, 9317, 50144.0)
(5, 9307, 54966.0)
(32, 9272, 38314.0)
(36, 9198, 44134.28571428572)
(21, 9104, 63264.0)
(37, 9065, 48692.57142857143)
(48, 9060, 34824.0)
(15, 9045, 34952.57142857143)
(72, 8994, 52979.142857142855)
(24, 8991, 49447.5)
(3, 8980, 63822.0)
(18, 8968, 44328.0)
(38, 8965, 39117.230769230766)
(31, 8930, 53805.0)
(77, 8909, 42764.0)
(9, 8899, 38018.666666666664)
(35, 8867, 31800.0)
(29, 8843, 41376.0)
(75, 8819, 63645.0)
(61, 8814, 53608.0)
(56, 8772, 44772.0)
(58, 8757, 23709.0)
(12, 8754, 57646.5)
(73, 8746, 53797.5)
(55, 8743, 44696.4)
(57,

### QUERY 4: List the districts of those clients who are in debt or haven’t paid their loan and see the demographic data of these districts

**Purpose:**  This query selects all the districts of those clients who are in dept or haven’t paid their loan and the demographic data them. This is useful for deciding on approval of any loan applications as it helps identifying any districts that are at higher risk of default. <br>
**Input parameters and conditions:** Datbase use can specify for which 'status' they want to see the data. For this specific query 'B' and 'D' are used.<br>
**Expected outputs:** From the 'Loans' table: 'account_id', 'B' and 'D' values from 'status', 'district_id' and all the columns from the 'Districts' table. <br>

In [ ]:
script = '''

WITH loans_list AS
    (SELECT account_id, status
    FROM Loans),

    loans_list_district AS
    (SELECT LL.account_id, LL.status, A.district_id
    FROM loans_list AS LL
    LEFT JOIN Accounts as A ON LL.account_id = A.account_id),

    loans_list_district_info AS
    (SELECT LLD.account_id, LLD.status, LLD.district_id, D.district_name, D.region, D.pop, D.pop_less_499,
    D.pop_less_1999, D.pop_less_9999, D.pop_greater_10000, D.city_no, D.urb_ratio, D.avg_sal, D.unemployment_95, D.unemployment_96,
    entreprenuers_per_1000, crimes_95, crimes_96
    FROM loans_list_district AS LLD
    LEFT JOIN Districts as D ON D.district_id = LLD.district_id)

SELECT * FROM loans_list_district_info
WHERE status = 'B' OR status = 'D';
'''

In [ ]:
result = conn.execute(script)
for row in result:
    print(row)

(19, 'B', 21, 'Tabor', 'south Bohemia', 103347, 87, 16, 7, 1, 7, 67, 9104, 1.51, 2.07, 123, 2299, 2354)
(37, 'D', 20, 'Strakonice', 'south Bohemia', 70646, 94, 14, 3, 1, 4, 58.4, 8547, 2.65, 3.64, 120, 1563, 1542)
(103, 'D', 44, 'Chrudim', 'east Bohemia', 105606, 77, 26, 7, 2, 7, 53, 8254, 2.79, 3.76, 97, 2166, 2325)
(347, 'B', 65, 'Znojmo', 'south Moravia', 114200, 101, 41, 4, 1, 4, 43.7, 8403, 5.74, 5.72, 105, 2157, 2718)
(426, 'D', 1, 'Hl.m. Praha', 'Prague', 1204953, 0, 0, 0, 1, 1, 100, 12541, 0.29, 0.43, 167, 85677, 99107)
(442, 'D', 54, 'Brno - mesto', 'south Moravia', 387570, 0, 0, 0, 1, 1, 100, 9897, 1.6, 1.96, 140, 18721, 18696)
(472, 'D', 8, 'Mlada Boleslav', 'central Bohemia', 112065, 95, 19, 7, 1, 8, 69.4, 11277, 1.25, 1.44, 127, 5179, 4987)
(790, 'B', 54, 'Brno - mesto', 'south Moravia', 387570, 0, 0, 0, 1, 1, 100, 9897, 1.6, 1.96, 140, 18721, 18696)
(808, 'D', 69, 'Jesenik', 'north Moravia', 42821, 4, 13, 5, 1, 3, 48.4, 8173, '?', 7.01, 124, '?', 1358)
(1106, 'B', 20, 'St

### QUERY 5: Count how many transactions where made from 1993 to 1998

**Purpose:** This query gives the database user an overview how many transactions were made from 1993 to 1998. This could be useful to track and analyse bank's growth by identifying new trends in the volume of transactions.<br>
**Input parameters and conditions:** There is no input parameters or conditions.<br>
**Expected outputs:** Year which is extracted from the'date' column from the Transactions table and the count for that value.<br>

In [ ]:
script = '''

SELECT substr(date, 1, 4) as year, COUNT(substr(date, 1, 4))
FROM Transactions
GROUP BY year


'''

In [ ]:
result = conn.execute(script)
for row in result:
    print(row)



('1993', 28205)
('1994', 91628)
('1995', 133022)
('1996', 196779)
('1997', 284409)
('1998', 322277)


### QUERY 6: Finds the total valuation of each accounts transactions

**Purpose:** This query allows the bank to identify valuable accounts. This could be useful either for identifying clients the bank wants to keep happy or for detecting accounts that may be engaging in fraudulent transactions.<br>
**Input parameters and conditions:** There is no input parameters or conditions.<br>
**Expected outputs:** The Account number and its corresponding sum of all passt transactions<br>

In [ ]:
q = (Transactions
     .select(Transactions.account_id, fn.SUM(Transactions.balance).alias('sum'))
     .group_by(Transactions.account_id)
     .order_by(fn.SUM(Transactions.balance).desc())).execute()

for values in range(0,20):
  print('Account:', q[values].account_id, 'Transaction Total:', round(q[values].sum,3))


Account: 96 Transaction Total: 50014838.6
Account: 2838 Transaction Total: 42892500.0
Account: 9265 Transaction Total: 41213152.8
Account: 1378 Transaction Total: 40115733.2
Account: 5952 Transaction Total: 39609532.1
Account: 3674 Transaction Total: 38684290.0
Account: 2762 Transaction Total: 38651987.2
Account: 9307 Transaction Total: 38139654.9
Account: 1779 Transaction Total: 37380623.0
Account: 8212 Transaction Total: 37267476.3
Account: 3260 Transaction Total: 37131261.3
Account: 8625 Transaction Total: 36163711.5
Account: 4321 Transaction Total: 35798009.0
Account: 1813 Transaction Total: 35766797.6
Account: 2932 Transaction Total: 35755256.7
Account: 2471 Transaction Total: 35688940.1
Account: 655 Transaction Total: 35677554.0
Account: 3558 Transaction Total: 35312949.6
Account: 10670 Transaction Total: 34991619.1
Account: 816 Transaction Total: 34729889.5


### Dynamic transaction query

**Purpose:** This query helps managing a database transaction. More specifically, this query inserts a new loan application into the dtabase, updates the status of the loan application, and then deletes any loan applications which the duration is less than 6 months. This query uses the BEGIN TRANSACTION, SAVE TRANSACTION, ROLLBACK TRANSACTION, and COMMIT TRANSACTION statements to manage the transaction and ensure that the database remains in a consistent state.<br>
**Input parameters and conditions:** No input parameters or conditions.
<br>
**Expected outputs:** No expected outputs.<br>

In [ ]:
script = '''

/* Check if there is an active transaction /*

IF @@TRANCOUNT > 0
BEGIN
    /* If there is an active transaction, commit it /*
    COMMIT TRANSACTION
END

/* Start a new transaction

BEGIN TRANSACTION


/* Insert a new loan application into the database /*

INSERT INTO Loans
VALUES (2020, 4484, 901212, 25672, 12, 2139, 'C')

/* Save the transaction /*
SAVE TRANSACTION savepoint1

/* Update the loan application status /*
UPDATE Loans
SET status = 'A'
WHERE loan_id = 2020

/* If the update fails, roll back to the saved transaction /*
IF @@ERROR <> 0
BEGIN
    ROLLBACK TRANSACTION savepoint1
END

/* If the update succeeds, delete loan applications which the duration is less than 6 months/*
ELSE
BEGIN
    DELETE FROM Loans
    WHERE duration < 6

    /* If the delete fails, roll back to the saved transaction /*
    IF @@ERROR <> 0
    BEGIN
        ROLLBACK TRANSACTION savepoint1
    END

    /* If the delete succeeds, commit the transaction /*
    ELSE
    BEGIN
        COMMIT TRANSACTION
    END
END



'''


In [ ]:
conn.execute(script)

# Views

### View 1: The total amount of loans each district has assumed

**Purpose:** This view displays the total amount of debt each district is in allowing the bank to further query this data to assist in the catagorisation of their clients and risk management. This can allow regional analysis and give the banks a good overview of all their total loans out. <br>
**Input parameters and conditions:** No input parameters or conditions.
<br>
**Expected outputs:** No expected outputs.<br>

In [ ]:
q = (Districts
     .select(Districts.district_name, fn.SUM(Loans.amount))
     .join(Accounts).where(Districts.district_id == Accounts.district_id)
     .join(Loans, JOIN.LEFT_OUTER).where(Loans.account_id == Accounts.account_id)
     .group_by(Districts.district_name))


In [ ]:
print(q)

SELECT "t1"."district_name", SUM("t2"."amount") FROM "districts" AS "t1" INNER JOIN "accounts" AS "t3" ON ("t3"."district_id" = "t1"."district_id") LEFT OUTER JOIN "loans" AS "t2" ON ("t2"."account_id" = "t3"."account_id") WHERE (("t1"."district_id" = "t3"."district_id") AND ("t2"."account_id" = "t3"."account_id")) GROUP BY "t1"."district_name"


In [ ]:
cursor.execute('''
CREATE VIEW district_loans AS
SELECT "t1"."district_name", SUM("t2"."amount") FROM "districts" AS "t1" INNER JOIN "accounts" AS "t3" ON ("t3"."district_id" = "t1"."district_id") LEFT OUTER JOIN "loans" AS "t2" ON ("t2"."account_id" = "t3"."account_id") WHERE (("t1"."district_id" = "t3"."district_id") AND ("t2"."account_id" = "t3"."account_id")) GROUP BY "t1"."district_name"
''')

cursor.execute('''
SELECT * FROM district_loans
''')
cursor.fetchall()

[('Benesov', 887952),
 ('Beroun', 1460796),
 ('Blansko', 1191024),
 ('Breclav', 1176792),
 ('Brno - mesto', 4049400),
 ('Brno - venkov', 1232232),
 ('Bruntal', 1277796),
 ('Ceska Lipa', 462684),
 ('Ceske Budejovice', 2010924),
 ('Cesky Krumlov', 584328),
 ('Cheb', 438816),
 ('Chomutov', 923916),
 ('Chrudim', 1431684),
 ('Decin', 509952),
 ('Domazlice', 397008),
 ('Frydek - Mistek', 2053752),
 ('Havlickuv Brod', 1193208),
 ('Hl.m. Praha', 12932412),
 ('Hodonin', 1191504),
 ('Hradec Kralove', 630672),
 ('Jablonec n. Nisou', 127200),
 ('Jesenik', 1938432),
 ('Jicin', 851364),
 ('Jihlava', 366096),
 ('Jindrichuv Hradec', 1339572),
 ('Karlovy Vary', 845196),
 ('Karvina', 3059820),
 ('Kladno', 1106520),
 ('Klatovy', 270876),
 ('Kolin', 1901160),
 ('Kromeriz', 960072),
 ('Kutna Hora', 2095980),
 ('Liberec', 971292),
 ('Litomerice', 1116228),
 ('Louny', 1540056),
 ('Melnik', 557796),
 ('Mlada Boleslav', 617520),
 ('Most', 615060),
 ('Nachod', 1768380),
 ('Novy Jicin', 694368),
 ('Nymburk', 100

### View 2: The total amount of loans each district has assumed

**Purpose:** This simple but useful view allows banks to easily see how many of each type of card they have active allowing them to understand what their clientele is more inclined towards. <br>
**Input parameters and conditions:** No input parameters or conditions.
<br>
**Expected outputs:** card_type view created & displayed.<br>

In [ ]:
q = (Credit_cards
     .select(Credit_cards.card_type, fn.COUNT(Credit_cards.card_type))
     .group_by(Credit_cards.card_type))

In [ ]:
print(q)

SELECT "t1"."card_type", COUNT("t1"."card_type") FROM "credit_cards" AS "t1" GROUP BY "t1"."card_type"


In [ ]:
cursor.execute('''
CREATE VIEW card_type AS
SELECT "t1"."card_type", COUNT("t1"."card_type")
FROM "credit_cards"
AS "t1" GROUP BY "t1"."card_type"
''')

cursor.execute('''
SELECT * FROM card_type
''')
cursor.fetchall()

[('classic', 659), ('gold', 88), ('junior', 145)]

### View 3: The value of each districts total transactions

**Purpose:** Similar to the previous view, this allows the bank to identify profitable districts that make frequent transactions, used in conjection with this view the bank can create risk portfolios to assess districts that create frequent and valuble transactions against how much they may be in debt to the bank. <br>
**Input parameters and conditions:** No input parameters or conditions.
<br>
**Expected outputs:** dis_trans view created in db & displayed.<br>

In [ ]:
q = (Transactions
     .select(Accounts.district_id, fn.SUM(Transactions.balance).alias('sum'))
     .join(Accounts, JOIN.LEFT_OUTER)
     .where(Transactions.account_id == Accounts.account_id)
     .group_by(Accounts.district_id)
     .order_by(fn.SUM(Transactions.balance).desc()))

In [ ]:
print(q)

SELECT "t1"."district_id", SUM("t2"."balance") AS "sum" FROM "transactions" AS "t2" LEFT OUTER JOIN "accounts" AS "t1" ON ("t2"."account_id" = "t1"."account_id") WHERE ("t2"."account_id" = "t1"."account_id") GROUP BY "t1"."district_id" ORDER BY SUM("t2"."balance") DESC


In [ ]:
cursor.execute('''
CREATE VIEW dis_trans AS
SELECT "t1"."district_id", SUM("t2"."balance")
AS "sum" FROM "transactions" AS "t2" LEFT OUTER JOIN "accounts" AS "t1" ON ("t2"."account_id" = "t1"."account_id")
WHERE ("t2"."account_id" = "t1"."account_id")
GROUP BY "t1"."district_id"
ORDER BY SUM("t2"."balance") DESC
''')

cursor.execute('''
SELECT * FROM dis_trans;
''')
cursor.fetchall()

[(1, 5156492016.799973),
 (70, 1400877923.3000073),
 (74, 1350076137.9999857),
 (54, 1046367366.8999981),
 (64, 891400073.899996),
 (68, 855400655.6999986),
 (72, 835893440.2999935),
 (52, 649505697.1000011),
 (5, 606824155.0000006),
 (38, 569737751.3000087),
 (66, 544078189.2999991),
 (47, 522945589.5000017),
 (73, 515978837.4000013),
 (19, 503458600.3999979),
 (75, 496756157.70000154),
 (76, 490080974.5999999),
 (60, 484878058.00000226),
 (33, 483745502.7999999),
 (31, 481214881.3000011),
 (36, 478952102.6000014),
 (53, 477547274.00000274),
 (55, 471119280.0000031),
 (16, 468796241.8000014),
 (59, 467628082.1000018),
 (8, 463162406.70000494),
 (61, 457647917.7999985),
 (48, 456273692.49999833),
 (6, 452621865.60000205),
 (9, 452072775.9999983),
 (41, 449336518.99999833),
 (50, 439739513.1999991),
 (11, 439637959.4000002),
 (15, 437858863.2999963),
 (40, 435414619.4999979),
 (63, 434000600.70000124),
 (10, 432835898.5999988),
 (24, 431176095.8000026),
 (28, 427841297.60000056),
 (21, 

### View 4: The number of accounts each district has and where they're located

**Purpose:** Identifies regions with the most active accounts. This can help banks identify possible fraud if used in conjunction with regional population data to identify discrepancies as well as honing in on more lucrative regions that interact more with the bank identifying regions the bank may want to pay special attention to or showing areas the bank can expand into for growth. <br>
**Input parameters and conditions:** No input parameters or conditions.
<br>
**Expected outputs:** dis_accounts view created in db and displayed.<br>

In [ ]:
q = (Districts
    .select(Districts.district_name, Districts.region, fn.COUNT(Accounts.account_id).alias('count'))
    .join(Accounts, JOIN.LEFT_OUTER)
    .where(Districts.district_id == Accounts.district_id)
    .group_by(Districts.district_name))

In [ ]:
cursor.execute('''
CREATE VIEW dis_accounts AS
SELECT "t1"."district_name", "t1"."region", COUNT("t2"."account_id") AS "count"
FROM "districts" AS "t1"
LEFT OUTER JOIN "accounts" AS "t2" ON ("t2"."district_id" = "t1"."district_id")
WHERE ("t1"."district_id" = "t2"."district_id")
GROUP BY "t1"."district_name"

''')

cursor.execute('''
SELECT * FROM dis_accounts;
''')
cursor.fetchall()

[('Benesov', 'central Bohemia', 42),
 ('Beroun', 'central Bohemia', 50),
 ('Blansko', 'south Moravia', 50),
 ('Breclav', 'south Moravia', 44),
 ('Brno - mesto', 'south Moravia', 128),
 ('Brno - venkov', 'south Moravia', 53),
 ('Bruntal', 'north Moravia', 43),
 ('Ceska Lipa', 'north Bohemia', 48),
 ('Ceske Budejovice', 'south Bohemia', 41),
 ('Cesky Krumlov', 'south Bohemia', 50),
 ('Cheb', 'west Bohemia', 50),
 ('Chomutov', 'north Bohemia', 39),
 ('Chrudim', 'east Bohemia', 50),
 ('Decin', 'north Bohemia', 49),
 ('Domazlice', 'west Bohemia', 36),
 ('Frydek - Mistek', 'north Moravia', 83),
 ('Havlickuv Brod', 'east Bohemia', 48),
 ('Hl.m. Praha', 'Prague', 554),
 ('Hodonin', 'south Moravia', 41),
 ('Hradec Kralove', 'east Bohemia', 49),
 ('Jablonec n. Nisou', 'north Bohemia', 43),
 ('Jesenik', 'north Moravia', 48),
 ('Jicin', 'east Bohemia', 41),
 ('Jihlava', 'south Moravia', 32),
 ('Jindrichuv Hradec', 'south Bohemia', 52),
 ('Karlovy Vary', 'west Bohemia', 42),
 ('Karvina', 'north Mor

### Close connections

In [ ]:
conn.close()
db.close()

True